# Hyperparameter Tuning for a PyTorch MNIST Model with Ray Tune

This notebook replaces a toy mathematical objective with a practical neural-network tuning experiment. We use the FashionMNIST architecture from the distributed-training notebook and ask Ray Tune to find an effective hidden-layer width and number of hidden layers.

The original model has two hidden layers of width 512:

```text
784 inputs -> 512 -> 512 -> 10 classes
```

We generalize that fixed architecture so Ray can explore widths of 128, 256, and 512 units and depths of one, two, and three hidden layers. The notebook then demonstrates two GPU execution strategies:

1. **Section I — one GPU per independent Tune trial.** Different hyperparameter configurations run on different GPUs when multiple devices are available.
2. **Section II — multiple GPUs for one trial.** Each configuration is trained by a Ray `TorchTrainer` worker group using PyTorch DistributedDataParallel (DDP).

## Learning objectives

- Convert a fixed PyTorch architecture into a configurable model.
- Define a categorical grid search with Ray Tune.
- Keep training and validation data separate during model selection.
- Reserve one full GPU for each independent Tune trial.
- Use `TorchTrainer` and DDP to train one trial across multiple GPUs.
- Aggregate validation metrics across distributed workers.
- Select the best trial using validation accuracy.
- Reconstruct the winning architecture from its configuration.

In [1]:
from pathlib import Path

import ray
import torch
import torch.distributed as dist
import torch.nn as nn
from ray import tune
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer
from ray.tune.schedulers import ASHAScheduler
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import v2

DATA_ROOT = Path("data").resolve()
RANDOM_SEED = 42

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

# Download once in the notebook process. Trials reuse these local files.
_ = datasets.FashionMNIST(
    root=DATA_ROOT,
    train=True,
    transform=transform,
    download=True,
)

# Section I — One GPU per independent hyperparameter trial

In this strategy, every Tune trial owns one complete model and reserves one full GPU. With four GPUs, up to four different configurations can train concurrently; with this machine's single GPU, trials run sequentially. Ray sets `CUDA_VISIBLE_DEVICES` for each trial, so the allocated physical device appears inside the trial as `cuda:0`.

## 1. Make the architecture configurable

`hidden_dim` controls the width of every hidden layer, while `num_hidden_layers` controls how many `Linear -> ReLU` blocks are created. For example:

| Configuration | Generated architecture |
|---|---|
| `hidden_dim=512`, `num_hidden_layers=1` | `784 -> 512 -> 10` |
| `hidden_dim=512`, `num_hidden_layers=2` | `784 -> 512 -> 512 -> 10` |
| `hidden_dim=256`, `num_hidden_layers=3` | `784 -> 256 -> 256 -> 256 -> 10` |

The two-layer, 512-unit configuration exactly reproduces the structure of the original model.

In [2]:
class NeuralNetwork(nn.Module):
    def __init__(self, hidden_dim: int, num_hidden_layers: int):
        super().__init__()

        if hidden_dim <= 0:
            raise ValueError("hidden_dim must be positive")
        if num_hidden_layers < 1:
            raise ValueError("num_hidden_layers must be at least 1")

        layers = []
        input_dim = 28 * 28

        for _ in range(num_hidden_layers):
            layers.extend([
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
            ])
            input_dim = hidden_dim

        layers.append(nn.Linear(input_dim, 10))

        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(*layers)

    def forward(self, x):
        return self.linear_relu_stack(self.flatten(x))


# Confirm that the original 512-by-2 architecture is generated correctly.
reference_model = NeuralNetwork(hidden_dim=512, num_hidden_layers=2)
print(reference_model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## 2. Define the Tune training function

Ray calls `train_model_on_one_gpu(config)` once per trial. The configuration determines the architecture and optimizer settings for that trial. The function fails explicitly if Tune schedules it without a CUDA device.

The 60,000 FashionMNIST training examples are split deterministically into 50,000 training and 10,000 validation examples. Hyperparameters are selected using validation accuracy—not training loss—because the goal is to identify the architecture that generalizes best to unseen examples.

`tune.report()` sends metrics to Ray after every epoch. This allows Tune to compare trials and allows the ASHA scheduler to stop weak configurations before they consume all available training time. Pinned host memory and non-blocking transfers are used to reduce CPU-to-GPU input overhead.

In [3]:
def train_model_on_one_gpu(config):
    if not torch.cuda.is_available():
        raise RuntimeError("This Tune trial requires one CUDA GPU")

    torch.manual_seed(RANDOM_SEED)
    device = torch.device("cuda:0")

    full_dataset = datasets.FashionMNIST(
        root=DATA_ROOT,
        train=True,
        transform=transform,
        download=False,
    )

    split_generator = torch.Generator().manual_seed(RANDOM_SEED)
    train_dataset, validation_dataset = random_split(
        full_dataset,
        [50_000, 10_000],
        generator=split_generator,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=512,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    model = NeuralNetwork(
        hidden_dim=config["hidden_dim"],
        num_hidden_layers=config["num_hidden_layers"],
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
    )

    for epoch in range(config["num_epochs"]):
        model.train()
        training_loss = 0.0

        for inputs, labels in train_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            predictions = model(inputs)
            loss = criterion(predictions, labels)
            loss.backward()
            optimizer.step()
            training_loss += loss.item() * inputs.size(0)

        model.eval()
        validation_loss = 0.0
        correct = 0

        with torch.inference_mode():
            for inputs, labels in validation_loader:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                predictions = model(inputs)
                loss = criterion(predictions, labels)
                validation_loss += loss.item() * inputs.size(0)
                correct += (predictions.argmax(dim=1) == labels).sum().item()

        tune.report({
            "epoch": epoch + 1,
            "training_loss": training_loss / len(train_dataset),
            "validation_loss": validation_loss / len(validation_dataset),
            "validation_accuracy": correct / len(validation_dataset),
        })

## 3. Configure and run one-GPU trials

The two architecture dimensions use `grid_search`, so Ray schedules all nine combinations:

```text
3 hidden dimensions × 3 layer counts = 9 trials
```

Learning rate, batch size, and epoch count are fixed so this example isolates the effect of model width and depth. Each trial reserves one full GPU and two CPUs. The maximum concurrency is the smaller of four and the number of GPUs detected by Ray. Therefore, this one-GPU machine runs one trial at a time, while a four-GPU node could run four configurations concurrently.

ASHA uses the reported validation accuracy to stop poorly performing trials after the grace period. This becomes especially valuable when searches contain many configurations or run for many epochs.

In [4]:
# Shut down a notebook-created runtime before starting a clean local instance.
ray.shutdown()
ray.init(include_dashboard=False, logging_level="ERROR")

available_gpus = int(ray.cluster_resources().get("GPU", 0))
available_cpus = int(ray.cluster_resources().get("CPU", 0))
if available_gpus < 1:
    raise RuntimeError("Section I requires at least one GPU visible to Ray")

max_parallel_gpu_trials = min(4, available_gpus, available_cpus // 2)
if max_parallel_gpu_trials < 1:
    raise RuntimeError("Each GPU trial requires two CPUs")
print(f"Ray detected {available_gpus} GPU(s)")
print(f"Concurrent one-GPU trials: {max_parallel_gpu_trials}")

search_space = {
    "hidden_dim": tune.grid_search([128, 256, 512]),
    "num_hidden_layers": tune.grid_search([1, 2, 3]),
    "learning_rate": 1e-3,
    "batch_size": 512,
    "num_epochs": 3,
}

scheduler = ASHAScheduler(
    time_attr="training_iteration",
    max_t=search_space["num_epochs"],
    grace_period=1,
    reduction_factor=2,
)

single_gpu_trainable = tune.with_resources(
    train_model_on_one_gpu,
    resources={"cpu": 2, "gpu": 1},
)

tuner = tune.Tuner(
    single_gpu_trainable,
    param_space=search_space,
    tune_config=tune.TuneConfig(
        metric="validation_accuracy",
        mode="max",
        scheduler=scheduler,
        max_concurrent_trials=max_parallel_gpu_trials,
    ),
    run_config=tune.RunConfig(
        name="fashion_mnist_one_gpu_per_trial",
        storage_path=str(Path("ray_results").resolve()),
        verbose=1,
    ),
)

single_gpu_results = tuner.fit()

## 4. Inspect the trials and rebuild the best model

The result grid contains one result per configuration. We sort trials by their final reported validation accuracy and use the same final-epoch rule to retrieve the winning configuration. Trials stopped early by ASHA are compared using the last epoch they completed.

In [5]:
results_table = single_gpu_results.get_dataframe()[[
    "config/hidden_dim",
    "config/num_hidden_layers",
    "validation_accuracy",
    "validation_loss",
    "training_iteration",
]].sort_values("validation_accuracy", ascending=False)

display(results_table)

best_result = single_gpu_results.get_best_result(
    metric="validation_accuracy",
    mode="max",
    scope="last",
)
best_config = best_result.config

print("Best configuration:")
print(f"  hidden_dim: {best_config['hidden_dim']}")
print(f"  num_hidden_layers: {best_config['num_hidden_layers']}")
print(
    "  validation_accuracy: "
    f"{best_result.metrics['validation_accuracy']:.4f}"
)

best_model = NeuralNetwork(
    hidden_dim=best_config["hidden_dim"],
    num_hidden_layers=best_config["num_hidden_layers"],
)
print("\nWinning architecture:")
print(best_model)

# Keep Ray running until Section II decides whether enough GPUs are available.

,config/hidden_dim,config/num_hidden_layers,validation_accuracy,validation_loss,training_iteration
5,512,2,0.8709,0.364080,3
8,512,3,0.8659,0.364243,3
4,256,2,0.8562,0.404700,3
2,512,1,0.8541,0.416677,3
0,128,1,0.8450,0.447193,3
1,256,1,0.8147,0.527533,1
7,256,3,0.8146,0.523512,1
6,128,3,0.7986,0.560598,1
3,128,2,0.7942,0.575392,1


Best configuration:
  hidden_dim: 512
  num_hidden_layers: 2
  validation_accuracy: 0.8709

Winning architecture:
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


# Section II — Multiple GPUs for one trial with DDP

Section I uses additional GPUs to evaluate more configurations concurrently. Section II uses additional GPUs to accelerate a single configuration. Every trial is a Ray `TorchTrainer` containing two GPU workers, and `prepare_model()` wraps the model with PyTorch DistributedDataParallel.

Each worker receives a different dataset shard through the distributed samplers installed by `prepare_data_loader()`. DDP averages gradients after every backward pass so all model replicas remain synchronized. The effective global training batch is:

```text
batch size per worker × DDP workers
```

Validation loss, correct predictions, and sample counts are summed across workers with `torch.distributed.all_reduce()` before metrics are reported. This avoids selecting a trial using only rank zero's validation shard.

> This machine exposes one GPU, so the execution cell below documents and constructs the two-GPU experiment but safely skips `fit()`. Run it on a machine or Ray cluster with at least two GPUs. On a multi-node cluster, `DATA_ROOT` must refer to storage visible from every worker node.

In [ ]:
def train_model_with_ddp(config):
    torch.manual_seed(RANDOM_SEED)

    context = ray.train.get_context()
    world_size = context.get_world_size()

    full_dataset = datasets.FashionMNIST(
        root=DATA_ROOT,
        train=True,
        transform=transform,
        download=False,
    )

    split_generator = torch.Generator().manual_seed(RANDOM_SEED)
    train_dataset, validation_dataset = random_split(
        full_dataset,
        [50_000, 10_000],
        generator=split_generator,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size_per_worker"],
        shuffle=True,
        num_workers=0,
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=512,
        shuffle=False,
        num_workers=0,
    )

    # These helpers add distributed samplers and move batches/model to
    # the GPU assigned to this worker.
    train_loader = ray.train.torch.prepare_data_loader(train_loader)
    validation_loader = ray.train.torch.prepare_data_loader(validation_loader)

    model = NeuralNetwork(
        hidden_dim=config["hidden_dim"],
        num_hidden_layers=config["num_hidden_layers"],
    )
    model = ray.train.torch.prepare_model(model)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
    )
    device = ray.train.torch.get_device()

    for epoch in range(config["num_epochs"]):
        if hasattr(train_loader.sampler, "set_epoch"):
            train_loader.sampler.set_epoch(epoch)

        model.train()
        local_training_loss = 0.0
        local_training_count = 0

        for inputs, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            predictions = model(inputs)
            loss = criterion(predictions, labels)
            loss.backward()
            optimizer.step()

            local_training_loss += loss.item() * inputs.size(0)
            local_training_count += inputs.size(0)

        model.eval()
        local_validation_loss = 0.0
        local_validation_count = 0
        local_correct = 0

        with torch.inference_mode():
            for inputs, labels in validation_loader:
                predictions = model(inputs)
                loss = criterion(predictions, labels)
                local_validation_loss += loss.item() * inputs.size(0)
                local_validation_count += inputs.size(0)
                local_correct += (
                    predictions.argmax(dim=1) == labels
                ).sum().item()

        # All workers receive the same globally aggregated statistics.
        totals = torch.tensor(
            [
                local_training_loss,
                local_training_count,
                local_validation_loss,
                local_validation_count,
                local_correct,
            ],
            dtype=torch.float64,
            device=device,
        )
        dist.all_reduce(totals, op=dist.ReduceOp.SUM)

        metrics = {
            "epoch": epoch + 1,
            "training_loss": (totals[0] / totals[1]).item(),
            "validation_loss": (totals[2] / totals[3]).item(),
            "validation_accuracy": (totals[4] / totals[3]).item(),
            "world_size": world_size,
            "global_batch_size": (
                config["batch_size_per_worker"] * world_size
            ),
        }

        # Because this function runs inside TorchTrainer, use train.report,
        # not tune.report. Every worker must call report the same number of times.
        ray.train.report(metrics)

## Configure the multi-GPU trial

`ScalingConfig(num_workers=2, use_gpu=True)` creates two training workers and reserves one GPU for each worker. Thus, this single DDP trial requires two GPUs. The trial retrains the architecture selected by Section I while distributing its data and gradient computation across both devices.

Ray Train 2.58 does not accept a v2 `TorchTrainer` directly as the trainable for `tune.Tuner`. Therefore, Section I performs the hyperparameter search with one-GPU Tune functions, and Section II passes the winning configuration to `TorchTrainer.fit()` for one multi-GPU DDP training run.

In [ ]:
GPUS_PER_DDP_TRIAL = 2
available_gpus = int(ray.cluster_resources().get("GPU", 0))

ddp_result = None

if available_gpus < GPUS_PER_DDP_TRIAL:
    print(
        f"Skipping Section II: detected {available_gpus} GPU(s), but each "
        f"DDP trial requires {GPUS_PER_DDP_TRIAL}."
    )
else:
    if "best_config" not in globals():
        raise RuntimeError("Run Section I before starting the DDP trial")

    ddp_loop_config = {
        "hidden_dim": best_config["hidden_dim"],
        "num_hidden_layers": best_config["num_hidden_layers"],
        "learning_rate": best_config["learning_rate"],
        "batch_size_per_worker": 256,
        "num_epochs": 3,
    }

    ddp_trainer = TorchTrainer(
        train_loop_per_worker=train_model_with_ddp,
        train_loop_config=ddp_loop_config,
        scaling_config=ScalingConfig(
            num_workers=GPUS_PER_DDP_TRIAL,
            use_gpu=True,
            resources_per_worker={"CPU": 1, "GPU": 1},
        ),
    )

    print("Starting one two-GPU DDP trial with:")
    print(ddp_loop_config)
    ddp_result = ddp_trainer.fit()
    print("Final DDP metrics:")
    print(ddp_result.metrics)

ray.shutdown()

## Conclusion

Ray Tune turns the fixed FashionMNIST network into a repeatable GPU architecture search. Section I assigns one full GPU to each independent trial, which maximizes search throughput when many GPUs are available. Section II takes the selected configuration and assigns several GPUs to one `TorchTrainer` trial with DDP, which is appropriate when one model is too expensive for a single device.

These strategies optimize different bottlenecks. One-GPU trials evaluate more configurations simultaneously; multi-GPU trials shorten an individual configuration's training time but reduce the number of trials that can run concurrently. A wider or deeper model also has greater representational capacity, more parameters, and more potential to overfit, so validation accuracy—not model size—drives selection.

This short search is intended for learning. For a stronger experiment, increase the epoch count, add learning rate and regularization to the search space, save checkpoints, evaluate the selected model once on the untouched FashionMNIST test set, repeat trials with multiple random seeds, and compare accuracy together with samples per second and GPU utilization.